<a href="https://colab.research.google.com/github/Carlos-V-V/Gen-AI-for-CCM-pipeline-prototypes/blob/main/v3_Full_MFG_HJB_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**DESCRIPTION OF THIS VERSION**

This copy of the pipeline notebook includes an implementation of the full MFG framework, with both OT-based and HJB-based regularizers. In this version, the OT regularizer is the Lagrangian cost (1/2)|v|^2 along the trajectory. We define the NNs, the ODE system, the cost functional J, and then the training loop, which trains our NNs with the optimization problem defined in the paper draft / slides. After training, then we can visualize the predicted cell trajectories in animation form, plot the behavior of the training loss over training epochs, save all the outputs as CSV files, and save the trained NN models.

This run of the pipeline assumes the system of equations:

dx_i/dt = V*p_i + Morse

dp_i/dt = 0

In [ ]:
!pip install torchdiffeq
!pip install geomloss

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchdiffeq import odeint  # from the `torchdiffeq' package
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cpu


We define the known kernels to use in this iteration. We define them like this so we can integrate the parameters into the optimizing pipeline, so that backpropagation flows through both the NN parameters (\phi) and the known kernel parameters (\theta). That is, the following is a trainable kernel model.  

For this example I will use only Morse potentials for dx/dt and a simple polarity term for dp/dt:

In [ ]:
class PosKernels(nn.Module):
  def __init__(self):
    super().__init__()
    self.A = nn.Parameter(torch.tensor(0.1))
    self.a = nn.Parameter(torch.tensor(1.0))
    self.R = nn.Parameter(torch.tensor(0.1))
    self.r = nn.Parameter(torch.tensor(0.5))
    #self.V = nn.Parameter(torch.tensor(1.0))


  def forward(self, x_ij, p_i):
    d = torch.norm(x_ij)
    epsilon = 1e-7  # Small epsilon to prevent division by zero
    return (-self.A*torch.exp(-d / (self.a + epsilon)) + self.R*torch.exp(-d / (self.r + epsilon)))*(x_ij / (d + epsilon)) #+ V*p_i  # Define assumed kernels

In [ ]:
class PolarityKernel(nn.Module):
    def __init__(self):
        super().__init__()
        self.g = nn.Parameter(torch.tensor(1.0)) # g is the parameter for the polarity equation, I'm just using one parameter for now

    def forward(self, x_ij, p_i, p_j):
        return 0  # Assumed polarity kernels

Now we define the NN for the value function U(t,x)

In [ ]:
class ValueNet(nn.Module):
  def __init__(self, input_dim, output_dim, hidden_dim):
    super().__init__()

    self.net = nn.Sequential(
        nn.Linear(input_dim, hidden_dim * 2), # Make first layer wider
        nn.Tanh(), # Or nn.ReLU()
        nn.Linear(hidden_dim * 2, hidden_dim * 2),
        nn.Tanh(), # Or nn.ReLU()
        nn.Linear(hidden_dim * 2, hidden_dim * 2),
        nn.Tanh(), # Or nn.ReLU()
        nn.Linear(hidden_dim * 2, hidden_dim * 2), # Another hidden layer
        nn.Tanh(), # Or nn.ReLU()
        nn.Linear(hidden_dim * 2, output_dim)
        )

  def forward(self, t, x):
      if t.ndim == 1:
          t = t[:, None]
      return self.net(torch.cat([t, x], dim=-1))

Now we define the right-hand side of the ODEs in Eq (6) using now the value function NN model and the known kernels we defined.

NOTE that the following code **does not include polarity dynamics**:

In [ ]:
class ODESystem(nn.Module): # Defines ODE system as a PyTorch module
  def __init__(self, known_kernel_x, value_net, known_kernel_p):
    super().__init__()
    self.known_kernel_x = known_kernel_x
    self.value_net = value_net
    self.known_kernel_p = known_kernel_p


  def forward(self, t, X):  # Defines the ODE rhs dX/dt = f(t,X), where X = (x1,...xN,p1,...,pN) is the current state of the system
    N = X.shape[0]//2 # Define number of cells here! It's the shape of X divided by 2, since X has both positions and polarities.
    positions = X[:N] # Extracts positions, each entry is a 2D position vector
    polarities = X[N:]

    positions = positions.requires_grad_(True)

    '''---------------------------------------------------------
    THE NEXT PART IS THE KNOWN VELOCITY PART (pre-existing code)
    ------------------------------------------------------------
    '''

    # Vectorized calculation of pairwise differences and distances
    x_i = positions.unsqueeze(1) # Shape: (N, 1, D)
    x_j = positions.unsqueeze(0) # Shape: (1, N, D)
    x_ij = x_i - x_j            # Shape: (N, N, D)
    d_ij = torch.norm(x_ij, dim=-1) # Shape: (N, N)

    epsilon = 1e-8

    A = self.known_kernel_x.A                       # Modify here depending on parameters
    a = self.known_kernel_x.a
    R = self.known_kernel_x.R
    r = self.known_kernel_x.r

    scalar_part = -A * torch.exp(-d_ij / a) + R * torch.exp(-d_ij / r)    # ***** MODIFY KERNELS HERE

    scalar_part.diagonal(0).fill_(0.0)  # Avoids unphysical self-interactions

    d_ij_unit_vec = x_ij / (d_ij.unsqueeze(-1) + epsilon) # Normalizing
    d_ij_unit_vec[torch.eye(N, dtype=torch.bool, device=d_ij.device)] = 0.0 # Ensures diagonal vectors are 0, so there's no self-interactions

    known_x_forces_all_pairs = scalar_part.unsqueeze(-1) * d_ij_unit_vec  # multiplies the unit vector to give it the direction
    v_known = known_x_forces_all_pairs.sum(dim=1) # Sums over j to get the total 'known' force on each particle i, and forms the vector of all of them
                                                  # Shape becomes (N, 2) for the 2-D forces

    '''----------------------------------------------
    NOW COMES THE LEARNED PART:   v^NN = - grad(U)
    We'll feed ValueNet the per-particle state [x_i, p_i]
    -------------------------------------------------
    '''

    x_in = torch.cat([positions, polarities], dim=-1) # shape (N, 4) for (positions , polarities)

    t_batch = t.expand(N)   # This makes a time vector so ValueNet can be called "per particle" in batch
                            # This is because ValueNet expects a batch of vectors "batched" along time
                            # Each entry is the same time t
    U = self.value_net(t_batch, x_in)   # Evaluates the ValueNet at the inputs

    grad_pos = torch.autograd.grad(outputs = U.sum(), inputs=positions, create_graph=True, allow_unused = False)[0] # Gradient of U wrt positions, shape (N,2)
                                                                                              # bc position is 2D, so this is actually a Jacobian

    v_theta = -grad_pos

    dX = v_known + v_theta

    '''----------------------------------------------------------
    For polarity dynamics, this is zero for this proof of concept
    -------------------------------------------------------------
    '''

    dP = torch.zeros_like(polarities)

    return torch.cat([dX, dP], dim=0) # Concatenates dX and dP


Now we define the **Hamiltonian**, this will be necessary for the HJB regularization term in the cost functional:

In [ ]:
def Hamiltonian(x,p):     # ***** CHECK THAT THIS DEFINITION IS CORRECT!!!
  # p is expected to be (Batch, Dim) or (Dim,). Sum over the last dimension (components of p).
  return 0.5*(p**2).sum(dim=-1, keepdim=True)

Now we define the **cost functional**:

In [ ]:
from geomloss import SamplesLoss

# We define the Wasserstein-2 divergence, in this case approximated by a Sinkhorn divergence (which is optimized and faster than computing the exact Wasserstein distance)
W2_dist = SamplesLoss(loss="sinkhorn", p=2, blur=0.5) # blur was 0.05 originally

# Both inputs must be (N, d) tensors (point clouds)

def cost_function_W2_OT_HJB(simulated_trajectory, target_snapshots, dX_traj_list, dP_traj_list, value_net, t_grid, x_grid, Hamiltonian, alphaD, alphaV, alpha1, alpha2):
  # simulated_trajectory: (steps, N, dim) - The full trajectory of simulated positions
  # target_snapshot: (N, dim) - The target point clouds for comparison
  # dX_traj_list, dP_traj_list: lists of (N, dim) tensors, containing the velocity vectors for OT regularization
  # N and dim are global variables (N=20, dim=2).

  total_w2_loss = 0.0
  num_sim_steps = simulated_trajectory.shape[0] # Number of time steps in the simulated trajectory

  # Part 1: Wasserstein-2 distance between simulated trajectory and target snapshots
  # Compare each simulated snapshot with the target snapshot, and add them all together ('path' formulation)
  for j in range(num_sim_steps):
    sim_j_positions = simulated_trajectory[j] # This is a (N, dim) tensor for the j-th time step
    target_snapshot = target_snapshots[j]
    current_w2 = W2_dist(sim_j_positions, target_snapshot) # Compare with the single target
    total_w2_loss += current_w2

  # Part 2: Optimal Transport-inspired Regularizer (running cost with Lagrangian)
  # This part relies on dX_traj_list and dP_traj_list which are lists of tensors (N, dim) for each timestep.
  total_ot_reg = 0.0
  for k in range(num_sim_steps): # Iterate through the collected dX/dP for each timestep
    total_ot_reg += (1/2)*torch.sum(dX_traj_list[k]**2) # Sum of squares of elements
    total_ot_reg += (1/2)*torch.sum(dP_traj_list[k]**2)

  # Part 3: HJB Regularizer
  # This function is nested and can access outer scope variables, but the arguments make it explicit:
  def HJBregularizer_internal(value_net_arg, t_grid_arg, x_grid_arg, Hamiltonian_arg, alpha1_arg, alpha2_arg,
                              simulated_trajectory_arg, target_snapshots_arg, num_sim_steps_arg):
    # N and dim are globally defined (N=20, dim=2).
    device = x_grid_arg.device

    t_colloc = t_grid_arg.clone().detach().requires_grad_(True) # (steps,)
    x_colloc = x_grid_arg.clone().detach().requires_grad_(True) # (steps, N, 2*dim)

    # Reshape for ValueNet batch input (ValueNet expects t: (B,1) or (B,), x: (B,d))
    t_batch_for_valuenet = t_colloc.unsqueeze(1).repeat(1, N).view(-1) # (steps*N,)
    x_batch_for_valuenet = x_colloc.view(-1, 2 * dim) # (steps*N, 2*dim)

    U = value_net_arg(t_batch_for_valuenet, x_batch_for_valuenet) # (steps*N, 1)

    # Compute U_t (time derivative of U)
    # The gradient is computed w.r.t. flattened t.
    U_t_flat = torch.autograd.grad(U.sum(), t_batch_for_valuenet, create_graph=True, allow_unused=True)[0]
    U_t_per_particle_per_timestep = U_t_flat.view(num_sim_steps_arg, N, 1) # (steps, N, 1)

    # Compute gradU (spatial gradient of U)
    gradU_flat = torch.autograd.grad(U.sum(), x_batch_for_valuenet, create_graph=True, allow_unused=True)[0]
    gradU_per_particle_per_timestep = gradU_flat.view(num_sim_steps_arg, N, 2 * dim) # (steps, N, 2*dim)

    # Compute Hamiltonian H
    # Hamiltonian(x, p) expects p as momentum and x as position.
    positions_for_H = x_colloc[:, :, :dim] # (steps, N, dim)
    momentum_for_H = gradU_per_particle_per_timestep[:, :, :dim] # Gradient w.r.t positions part of state (steps, N, dim)

    # Flatten for Hamiltonian which expects (Batch, D)
    momentum_for_H_flat = momentum_for_H.reshape(-1, dim) # (steps*N, dim)
    positions_for_H_flat = positions_for_H.reshape(-1, dim) # (steps*N, dim)

    H_flat = Hamiltonian_arg(positions_for_H_flat, momentum_for_H_flat) # (steps*N, 1)
    H_per_particle_per_timestep = H_flat.view(num_sim_steps_arg, N, 1) # (steps, N, 1)

    # Compute Interaction Cost I (for HJB regularizer)
    # I_per_timestep will be a scalar for each timestep, representing W2 distance from sim to target.
    I_per_timestep_values = torch.zeros(num_sim_steps_arg, device=device)
    for j in range(num_sim_steps_arg):
        I_per_timestep_values[j] = W2_dist(simulated_trajectory_arg[j], target_snapshots[j])

    # Replicate I for each particle in each timestep to match shape (steps, N, 1)
    I_per_particle_per_timestep = I_per_timestep_values.unsqueeze(1).repeat(1, N).unsqueeze(-1) # (steps, N, 1)

    # HJB residual: U_t - H + I
    HJB_residual = U_t_per_particle_per_timestep - H_per_particle_per_timestep + I_per_particle_per_timestep

    # HJB cost
    HJB_cost = alpha1_arg * torch.abs(HJB_residual).mean() # Mean over all steps and particles

    # Terminal cost term in HJB regularizer:
    xT_state = x_colloc[-1] # Last timeframe's full state (N, 2*dim)
    tT = t_colloc[-1] # Last timeframe (scalar)

    # Evaluate ValueNet at terminal time for all particles
    tT_batch = tT.expand(N) # (N,)
    xT_batch = xT_state # (N, 2*dim)
    UT_terminal = value_net_arg(tT_batch, xT_batch) # (N, 1)

    # MT is W2_dist(xT, Target_pos)
    xT_positions = xT_state[:, :dim] # Positions from last state (N, dim)
    MT_terminal = W2_dist(X_pred[-1], target_snapshots[-1]) # Scalar W2 distance

    HJB_terminal_cost = alpha2_arg * torch.abs(UT_terminal.mean() - MT_terminal) # Mean of UT and compare with MT

    total_hjb_reg = HJB_cost + HJB_terminal_cost
    return total_hjb_reg

  # Call the internal HJBregularizer function
  total_hjb_reg = HJBregularizer_internal(value_net, t_grid, x_grid, Hamiltonian, alpha1, alpha2,
                                          simulated_trajectory, target_snapshot, num_sim_steps)

  return alphaD*total_w2_loss / X_pred.shape[0] + alphaV*total_ot_reg + total_hjb_reg

We define the initial conditions, and the target dataset. For this, I upload a CSV file of a simulated dataset.

In [ ]:
import pandas as pd

from google.colab import files
uploaded = files.upload()

Saving (Run=2.1)_V=0.025_A=0.025_R=0.0375_a=0.25_r=0.125.csv to (Run=2.1)_V=0.025_A=0.025_R=0.0375_a=0.25_r=0.125 (3).csv


In [ ]:
df = pd.read_csv('(Run=2.1)_V=0.025_A=0.025_R=0.0375_a=0.25_r=0.125.csv', header=None)

In [ ]:
# We extract the first row of the dataset, i.e. the initial positions and polarities
row0 = df.iloc[0].values  # shape (80,1)

# We split into positions and polarities:
x0_flat = row0[:40]  # first 40 entries
p0_flat = row0[40:]  # next 40 entries

x_init_np = x0_flat.reshape(20, 2)  # shape (20, 2)
p_init_np = p0_flat.reshape(20, 2)  # shape (20, 2)

# Convert them to PyTorch tensors:
x_init = torch.tensor(x_init_np, dtype=torch.float32)  # shape (20, 2)
p_init = torch.tensor(p_init_np, dtype=torch.float32)  # shape (20, 2)

Define the "target" dataset as a tensor with shape (timeframes, 2*N, dim) comprised of the 'observed' synthetic data, with all the positions and then all the polarities for each timeframe.

In [ ]:
data_np = df.values  # Convert DataFrame to a NumPy array
# Reshape the data: 200 timeframes, 40 entities (positions + polarities), 2 dimensions (x,y or p_x, p_y)
target_snapshots_full = torch.tensor(data_np, dtype=torch.float32).reshape(200, 40, 2)
target_snapshots = target_snapshots_full[:50]
print(target_snapshots.shape)

torch.Size([50, 40, 2])


Now we initialize everything and train:

In [ ]:
from re import X

N = 20
T_final = 250 # Final time for the simulation, MY DATA HAS T_final = 1000 with 200 timeframes, with time_step = 5
steps = 50 # Number of time steps, Michael used 200 steps
num_epochs = 100  # Number of training EPOCHS, EVENTUALLY DO 100-1000

# Dimensions for the NN models:
input_dim = 3 # This is the input dimension for *each pairwise interaction* (r, p_i)
hidden_dim = 64 # Lower to 32 if it takes too long
output_dim = 2 # Output dimension is 2 for a 2D force

value_input_dim = 1 + 4  # U input is [t] + [x,p], where x, p are 2D each
value_net = ValueNet(input_dim = value_input_dim, output_dim = 1, hidden_dim = hidden_dim)

# Initialize known position dynamics
known_x = PosKernels()

# And polarity dynamics
known_p = PolarityKernel()

# ODE system
ode_func = ODESystem(known_kernel_x = known_x, value_net = value_net, known_kernel_p = known_p)

# Define parameter groups with different learning rates

LR_NN = 1e-3
LR_known = 1e-3

param_groups = [
    {'params': value_net.parameters(), 'lr': LR_NN},
    {'params': known_x.parameters(), 'lr': LR_known},
    {'params': known_p.parameters(), 'lr': LR_known}
]


# Initial condition:
X0 = torch.cat([x_init, p_init], dim=0) # Change to match initial conditions in the dataset, and figure out why "dim=0"

# Time points:
t = torch.linspace(0, T_final, steps)

# Optimizer
optimizer = optim.Adam(param_groups)

In [ ]:
# To run the pipeline on the GPU / CPU:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# These MUST be moved:
X0 = X0.to(device)
target_snapshots = target_snapshots.to(device)
t = t.to(device)
value_net = value_net.to(device)
ode_func = ode_func.to(device)


Below we initialize the weights / biases of the InteractionNet NN to smaller values. This helps control the initial magnitude of the unknown forces.

In [ ]:
def initialize_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight, gain=0.1) # Using Xavier uniform with a small gain
        if m.bias is not None:
            nn.init.constant_(m.bias, 0) # Initialize biases to zero

value_net.apply(initialize_weights)

print("Neural network weights initialized.")

Neural network weights initialized.


Extract the initial parameter values:

In [ ]:
for name, param in known_x.named_parameters():
    print(f"{name}: {param.data}")

A: 0.10000000149011612
a: 1.0
R: 0.10000000149011612
r: 0.5


We upload the trained models to keep training them (ignore this if you are training new models from scratch)

In [ ]:
files.upload()
files.upload()
files.upload()

**ONLY** run the following cell if loading saved models to **continue training**. Ignore if training new models from scratch.

In [ ]:
''' ONLY run this cell if loading saved models to continue training '''

# We create new instances of the NNs above, with the same input_dim, hidden_dim and output_dim
loaded_value_net = ValueNet(input_dim = value_input_dim, output_dim = 1, hidden_dim = hidden_dim)
loaded_known_x = PosKernels()
loaded_known_p = PolarityKernel()

# Load the saved state dictionaries
loaded_value_net.load_state_dict(torch.load('value_net_model_0219_1.pth'))
loaded_known_x.load_state_dict(torch.load('known_x_model_0219_1.pth'))
loaded_known_p.load_state_dict(torch.load('known_p_model_0219_1.pth'))

# Move the loaded models to the appropriate device (CPU or GPU)
loaded_value_net.to(device)
loaded_known_x.to(device)
loaded_known_p.to(device)


PolarityKernel()

The next cell tests how long it takes to integrate the NN-augmented ODE system. Here we re-define the ValueNet neural network within the cell because, for some reason, it was not recognizing it before.

In [ ]:
import time
import torch.nn as nn
start = time.time()

# Re-define ValueNet class here to ensure it is correctly loaded
class ValueNet(nn.Module):
  def __init__(self, input_dim, output_dim, hidden_dim):
    super().__init__()

    self.net = nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.Tanh(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.Tanh(),
        nn.Linear(hidden_dim, output_dim)
        )

  def forward(self, t, x):
      # t: (B,1) or (B,), x: (B,d)
      if t.ndim == 1:
          t = t[:, None]
      return self.net(torch.cat([t, x], dim=-1))  # (B,1)

# Re-instantiate value_net and ode_func to ensure correct forward method is recognized
# These variables (value_input_dim, output_dim, hidden_dim, known_x, known_p, device)
# are assumed to be defined in previous cells.
value_net = ValueNet(input_dim = value_input_dim, output_dim = 1, hidden_dim = hidden_dim)
ode_func = ODESystem(known_kernel_x = known_x, value_net = value_net, known_kernel_p = known_p)

# Apply weight initialization and move to device again for the new instances
value_net.apply(initialize_weights)

value_net = value_net.to(device)
ode_func = ode_func.to(device)

# If I encounter this error again:
# OutOfMemoryError: CUDA out of memory indicates that the GPU ran out of memory during the ODE integration.
# This is likely due to the memory required for the vectorized ODESystem and the ODE solver, which scales with N*N.
# To fix this, try reducing the number of cells (N) or the hidden dimension of the neural networks (hidden_dim) in cell jBfjnsaIry88.

X_pred = odeint(ode_func, X0, t, method='dopri5', rtol=1e-3, atol=1e-4) # Relaxed tolerances

print("Integration time:", time.time() - start)


Integration time: 3.467008352279663


Below is a **protoype** of the new training loop. Use the following cell to **test changes only**.

In [ ]:
# Test to see how long it takes to run one epoch of training:

import time
start = time.time()

# Define dim, alpha1, alpha2
dim = 2 # Dimension of position/polarity vectors
alphaD = 1
alphaV = 1e-3
alpha1 = 1e-3 # Placeholder regularization parameters
alpha2 = 1e-3

optimizer.zero_grad()
X_pred = odeint(ode_func, X0, t, method='dopri5', rtol=1e-3, atol=1e-4)
Pred_pos = X_pred[:, :N] # Corrected slicing for positions over all time steps
Pred_pol = X_pred[:, N:] # Corrected slicing for polarities over all time steps

''' We extract the dX, dP information for the OT regularizer '''
dX_traj = []
dP_traj = []

for i in range(len(X_pred)):
  # Corrected: Pass t[i] (a tensor) instead of i (an int) to ode_func
  dXdP = ode_func(t[i], X_pred[i])
  # Num is N, which is already defined as 20.
  dX_current = dXdP[:N] # Using global N
  dP_current = dXdP[N:] # Using global N

  dX_traj.append(dX_current)
  dP_traj.append(dP_current)

''' We prepare the collocation points for the HJB regularizer '''

# Refactored State creation to be a single tensor for efficiency and correct input for ValueNet
# State should be (steps, N, 2*D) -> (steps, N, 4) if D=2
State = torch.cat([Pred_pos, Pred_pol], dim=-1) # Shape: (steps, N, 4)

t_grid = t # Global time tensor for HJB regularizer
x_grid = State # State for HJB regularizer

''' Now we calculate the loss (a.k.a. cost): '''
# Pass dX_traj and dP_traj lists to the cost function
loss = cost_function_W2_OT_HJB(X_pred, target_snapshots, dX_traj, dP_traj, value_net, t_grid, x_grid, Hamiltonian, alphaD, alphaV, alpha1, alpha2)
loss.backward()
optimizer.step()

print("One epoch of training:", time.time() - start , f"Loss: {loss.item()}")


One epoch of training: 3.2967376708984375 Loss: 2.2794504165649414


Here now we do the full training, training for a total of "num_epochs" epochs.

In [ ]:
''' ONLY run the following lines if you are continuing to train models that had already been pre-trained and loaded '''
value_net = loaded_value_net
known_x = loaded_known_x
known_p = loaded_known_p

# Re-initialize the ODE system with the loaded and assigned models
# This makes sure that the ode_func uses the parameters from the loaded models.
ode_func = ODESystem(known_kernel_x = known_x, value_net = value_net, known_kernel_p = known_p)

# Move the re-initialized ode_func to the correct device
ode_func = ode_func.to(device)

print("Models assigned and ODE system re-initialized for continued training.")

Models assigned and ODE system re-initialized for continued training.


In [ ]:
num_epochs = 500

alphaD = 100     # Was 1e-3 before. Unweighted divergence-based costs were initially ~ 9400, so we wanna regularize everything to be of order ~ 10
alphaV = 1e0   # Was 10 before. Unweighted OT / action-costs were initially ~ 0.36
alpha1 = 5e1   # Unweighted HJB-reg costs were initially ~ 47
alpha2 = 5e1    # Unweighted U terminal costs were initially ~ 145

LR_NN = 1e-6
LR_known = 1e-6

rtol = 1e-2     # Originally 1e-3
atol = 1e-2     # Originally 1e-4

# Automate the training loop for num_epochs times
import time

optimizer = torch.optim.AdamW(
    [{'params': value_net.parameters(), 'lr': LR_NN, 'weight_decay': 1e-7},     # weight decay was originally 1e-4
     {'params': known_x.parameters(),   'lr': LR_known, 'weight_decay': 0.0}]
)

torch.nn.utils.clip_grad_norm_(value_net.parameters(), 1.0)
torch.nn.utils.clip_grad_norm_(known_x.parameters(), 1.0)
torch.nn.utils.clip_grad_norm_(known_p.parameters(), 1.0)

ode_func = ODESystem(known_kernel_x=known_x, value_net=value_net, known_kernel_p=known_p)
ode_func.to(device)

# Re-define W2_dist for self-containment within this cell
W2_dist = SamplesLoss(loss="sinkhorn", p=2, blur=0.05)

total_training_start_time = time.time()

# num_epochs, N, dim, alpha1, alpha2, optimizer, ode_func, X0, t,
# value_net, Hamiltonian, Target_pos are assumed to be defined globally

print(f"Starting training for {num_epochs} epochs...")
for epoch in range(num_epochs):
  epoch_start_time = time.time() # Start timer for this epoch

  optimizer.zero_grad()
  X_pred = odeint(ode_func, X0, t, method='dopri5', rtol=rtol, atol=atol)
  Pred_pos = X_pred[:, :N]
  Pred_pol = X_pred[:, N:]

  # Extract dX, dP information for the OT regularizer
  dX_traj = []
  dP_traj = []
  for i in range(len(X_pred)):
    dXdP = ode_func(t[i], X_pred[i])
    dX_current = dXdP[:N]
    dP_current = dXdP[N:]
    dX_traj.append(dX_current)
    dP_traj.append(dP_current)

  # Prepare collocation points for the HJB regularizer
  State = torch.cat([Pred_pos, Pred_pol], dim=-1) # Shape: (steps, N, 4)
  t_grid = t
  x_grid = State

  # Calculate the loss (a.k.a. cost)
  loss = cost_function_W2_OT_HJB(X_pred, target_snapshots, dX_traj, dP_traj, value_net, t_grid, x_grid, Hamiltonian, alphaD, alphaV, alpha1, alpha2)
  loss.backward()
  optimizer.step()

  print(f"Epoch {epoch+1}/{num_epochs} | Time: {time.time() - epoch_start_time:.4f}s | Loss: {loss.item():.4f}")

total_training_end_time = time.time()
print(f"Total training for {num_epochs} epochs completed in {total_training_end_time - total_training_start_time:.4f}s")

Starting training for 500 epochs...
Epoch 1/500 | Time: 1.0145s | Loss: 4.3564
Epoch 2/500 | Time: 1.1022s | Loss: 4.3892
Epoch 3/500 | Time: 1.0448s | Loss: 4.3884
Epoch 4/500 | Time: 1.0612s | Loss: 4.3731
Epoch 5/500 | Time: 1.0392s | Loss: 4.3519
Epoch 6/500 | Time: 1.0543s | Loss: 4.3595
Epoch 7/500 | Time: 1.0199s | Loss: 4.3540
Epoch 8/500 | Time: 1.0636s | Loss: 4.3597
Epoch 9/500 | Time: 1.6160s | Loss: 4.3622
Epoch 10/500 | Time: 1.4569s | Loss: 4.3581
Epoch 11/500 | Time: 0.9788s | Loss: 4.3518
Epoch 12/500 | Time: 1.0278s | Loss: 4.3539
Epoch 13/500 | Time: 1.1955s | Loss: 4.3506
Epoch 14/500 | Time: 1.0664s | Loss: 4.3499
Epoch 15/500 | Time: 1.1206s | Loss: 4.3531
Epoch 16/500 | Time: 1.0548s | Loss: 4.3518
Epoch 17/500 | Time: 1.0834s | Loss: 4.3524
Epoch 18/500 | Time: 0.9784s | Loss: 4.3521
Epoch 19/500 | Time: 0.9888s | Loss: 4.3505
Epoch 20/500 | Time: 1.3136s | Loss: 4.3504
Epoch 21/500 | Time: 1.4995s | Loss: 4.3503
Epoch 22/500 | Time: 1.1719s | Loss: 4.3493
Epoch

In [ ]:
for name, param in known_x.named_parameters():
    print(f"{name}: {param.data}")

A: 0.044976260513067245
a: 0.7445380091667175
R: 0.04493896663188934
r: 0.7001979947090149


Animation code:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import numpy as np # Import numpy for min/max calculations

# Ensure N is defined (from previous cells)
# N = 20
# X_pred is the tensor from the last odeint call (from training or simulation)
# It has shape (steps, 2*N, dim)

# Extract positions from X_pred
# Assuming X_pred is the output of the last training loop, it has shape (steps, 2*N, dim)
# The first N entries are positions, next N are polarities.
positions_pred = X_pred[:, :N, :] # Shape (steps, N, dim)

# Extract positions from target_snapshots
# target_snapshots has shape (timeframes, 2*N, dim)
target_positions = target_snapshots[:, :N, :] # Shape (timeframes, N, dim)

# Convert positions to numpy for plotting
positions_np = positions_pred.cpu().detach().numpy()
target_positions_np = target_positions.cpu().detach().numpy()

# Determine overall min/max for plot limits, considering both predicted and target data
all_x_coords = np.concatenate((positions_np[:, :, 0].flatten(), target_positions_np[:, :, 0].flatten()))
all_y_coords = np.concatenate((positions_np[:, :, 1].flatten(), target_positions_np[:, :, 1].flatten()))

min_x, max_x = all_x_coords.min() - 0.5, all_x_coords.max() + 0.5
min_y, max_y = all_y_coords.min() - 0.5, all_y_coords.max() + 0.5


# Set up the figure and axes
fig, ax = plt.subplots(figsize=(10, 10)) # Increased figure size for better visibility
ax.set_xlim(min_x, max_x)
ax.set_ylim(min_y, max_y)
ax.set_title('Predicted vs. Target Cell Positions Over Time')
ax.set_xlabel('X-coordinate')
ax.set_ylabel('Y-coordinate')
ax.set_aspect('equal', adjustable='box')
plt.grid(True)

# Initialize the plot with the first frame's data
line_pred, = ax.plot([], [], 'o', markersize=5, color='blue', label='Predicted')
line_target, = ax.plot([], [], 'o', markersize=5, color='red', label='Target')
ax.legend()

def init():
    line_pred.set_data([], [])
    line_target.set_data([], [])
    return line_pred, line_target, # Return all artists that will be updated

def animate(i):
    # Update the data for each frame for predicted trajectory
    x_coords_pred = positions_np[i, :, 0]
    y_coords_pred = positions_np[i, :, 1]
    line_pred.set_data(x_coords_pred, y_coords_pred)

    # Update the data for each frame for target trajectory
    x_coords_target = target_positions_np[i, :, 0]
    y_coords_target = target_positions_np[i, :, 1]
    line_target.set_data(x_coords_target, y_coords_target)

    return line_pred, line_target,

# Create the animation
# frames: number of time steps (length of the first dimension of positions_np)
# interval: delay between frames in ms
# blit=True means only re-draw the parts that have changed.
ani = animation.FuncAnimation(fig, animate, frames=min(positions_np.shape[0], target_positions_np.shape[0]), interval=100, blit=True)

# Save the animation as a GIF
gif_path = 'predicted_vs_target_cell_movement.gif'
ani.save(gif_path, writer='pillow', fps=2) # fps controls the speed of the animation

print(f"Animation saved to {gif_path}")

# Display the animation in the notebook (optional)
plt.close(fig) # Close the static plot window
HTML(ani.to_jshtml())

Animation saved to predicted_vs_target_cell_movement.gif


In [ ]:
from google.colab import files
files.download("predicted_vs_target_cell_movement.gif")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Now we extract the learned **v_NN** vectors for future analysis:

In [ ]:
def compute_v_theta_positions(value_net, t_scalar, positions, polarities):
    """
    Returns v_theta for POSITIONS only: shape (N,2)
    where v_theta = -∇_x U(t, [x,p]).
    """
    # Need grads wrt positions to compute ∇_x U
    pos_req = positions.detach().clone().requires_grad_(True)
    pol_det = polarities.detach()

    x_in = torch.cat([pos_req, pol_det], dim=-1)        # (N,4)
    t_batch = t_scalar.expand(pos_req.shape[0])         # (N,)

    U = value_net(t_batch, x_in)                        # (N,1) or (N,)
    if U.ndim == 1:
        U = U.unsqueeze(-1)

    grad_pos = torch.autograd.grad(
        outputs=U.sum(),
        inputs=pos_req,
        create_graph=False,   # logging only
        allow_unused=False
    )[0]  # (N,2)

    return -grad_pos


def integrate_and_log_vtheta_wide_csv(
    ode_func,          # trained ODESystem
    value_net,         # trained ValueNet (same as ode_func.value_net ideally)
    X0,                # (2N,2)
    t_grid,            # (steps,)
    csv_path="v_theta_wide.csv",
    method="dopri5",
    rtol=1e-3,
    atol=1e-4,
    include_time=True,
):
    """
    Produces CSV where each row corresponds to one timestep.
    Each row contains flattened v_theta_full of shape (2N,2) -> length 4N.

    v_theta_full = [v_theta_positions; zeros_for_polarities]  # (2N,2)
    """

    ode_func.eval()
    value_net.eval()

    # Integrate
    # X_traj: (steps, 2N, 2)
    X_traj = odeint(ode_func, X0, t_grid, method=method, rtol=rtol, atol=atol)

    steps = X_traj.shape[0]
    N = X0.shape[0] // 2

    # Build column names: flatten (2N,2) in row-major order
    # index order: [0,0], [0,1], [1,0], [1,1], ..., [2N-1,1]
    colnames = []
    if include_time:
        colnames.append("time")

    for idx in range(2 * N):
        colnames.append(f"vtheta_{idx}_dim0")
        colnames.append(f"vtheta_{idx}_dim1")

    data_rows = []

    for k in range(steps):
        t_k = t_grid[k]
        X_k = X_traj[k]            # (2N,2)
        pos = X_k[:N]              # (N,2)
        pol = X_k[N:]              # (N,2)

        # v_theta for positions (N,2)
        vtheta_pos = compute_v_theta_positions(value_net, t_k, pos, pol)

        # Full v_theta aligned with state layout (2N,2): bottom half zeros
        vtheta_full = torch.cat([vtheta_pos, torch.zeros_like(pol)], dim=0)  # (2N,2)

        # Flatten to length 4N
        flat = vtheta_full.reshape(-1).detach().cpu().tolist()

        if include_time:
            row = [float(t_k.detach().cpu().item())] + flat
        else:
            row = flat

        data_rows.append(row)

    df = pd.DataFrame(data_rows, columns=colnames)
    df.to_csv(csv_path, index=False)
    return df

In [ ]:
df = integrate_and_log_vtheta_wide_csv(
    ode_func=ode_func,
    value_net=value_net,
    X0=X0,
    t_grid=t,
    csv_path="v_theta_wide.csv",
    method="dopri5",
    rtol=1e-3,
    atol=1e-4,
    include_time=True,
)

print(df.shape)     # (steps, 1 + 4N) if include_time=True
print(df.head())

(20, 81)
        time  vtheta_0_dim0  vtheta_0_dim1  vtheta_1_dim0  vtheta_1_dim1  \
0   0.000000      -0.018527       0.015464      -0.005161       0.037305   
1   5.263158       0.020503      -0.000468       0.024965       0.008869   
2  10.526316       0.013953       0.002757       0.017744       0.016331   
3  15.789474       0.019681       0.002196       0.022052       0.012104   
4  21.052631       0.022069       0.003425       0.023206       0.010489   

   vtheta_2_dim0  vtheta_2_dim1  vtheta_3_dim0  vtheta_3_dim1  vtheta_4_dim0  \
0       0.092759      -0.076065      -0.010177       0.030291      -0.006598   
1       0.034765      -0.019702       0.023080       0.005775       0.023337   
2       0.046116      -0.031112       0.017200       0.010803       0.018467   
3       0.042742      -0.026096       0.021730       0.008001       0.022679   
4       0.036646      -0.020117       0.023098       0.007538       0.023738   

   ...  vtheta_35_dim0  vtheta_35_dim1  vtheta_36_dim

In [ ]:
files.download("v_theta_wide.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# We extract the final optimized parameters from known_x
optimized_known_x_params = known_x.parameters()

print("Optimized parameters:")
param_data = {}
for name, param in known_x.named_parameters():
    if param.requires_grad:
        print(name, param.data)
        param_data[name] = param.data.item() # Store parameter name and value

# Save parameters to a txt file
with open("Learned_kernel_params.txt", "w") as f:
    for name, value in param_data.items():
        f.write(f"{name}: {value}\n")

print("Optimized parameters saved to Learned_kernel_params.txt")

Optimized parameters:
A tensor(0.0750)
a tensor(0.8647)
R tensor(0.0880)
r tensor(0.5861)
Optimized parameters saved to Learned_kernel_params.txt
